# CIK Behavior Analysis

Profile every institutional investor (CIK) across all quarters in `normalized_holdings`, compute behavioral metrics, and produce a tunable filter that returns the CIKs matching user-defined thresholds.

**Metrics computed per CIK (aggregated over all quarters they appear):**

| Group | Metric | Meaning |
|---|---|---|
| Activity | `n_quarters` | quarters the CIK files |
| Activity | `first_quarter`, `last_quarter` | lifespan |
| Size | `aum_mean`, `aum_median` | avg portfolio USD value |
| Size | `aum_log_std` | volatility of log(AUM) qoq |
| Size | `aum_cagr` | AUM growth rate first→last |
| Breadth | `n_holdings_mean`, `n_holdings_std` | number of positions |
| Concentration | `hhi_mean` | Σ wᵢ² — 1 = single stock, 1/N = uniform |
| Concentration | `top5_weight_mean` | share in top-5 holdings |
| Turnover | `turnover_mean` | ½·Σ|Δwᵢ| per quarter (paper definition) |
| Turnover | `turnover_std` | stability of turnover |
| Churn | `open_rate_mean` | fraction of positions newly opened each qtr |
| Churn | `close_rate_mean` | fraction fully closed each qtr |
| Churn | `avg_holding_duration` | mean # quarters a position is held |

Use the **Filter** cell near the bottom to set thresholds and retrieve CIK lists.

## Setup

In [6]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path)
        print("Loaded .env from:", env_path.resolve())
        break

def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if (p / "ETL").is_dir():
            return p
    return cwd.parent.parent

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ETL.gnn_db_pipeline.config import TARGET_DB, TGT_NORMALIZED_HOLDINGS, TGT_CIK_AUM
from ETL.gnn_db_pipeline.db_connector import ConfigurablePostgresHandler

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)

handler = ConfigurablePostgresHandler(TARGET_DB)
handler.connect()
print("Connected to", TARGET_DB)

2026-04-21 12:21:08 - ETL_Pipeline - INFO - Connected to PostgreSQL: postgres@127.0.0.1:5432/13FGNN


Loaded .env from: C:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\.env
Connected to 13FGNN


## Load all holdings

In [7]:
import time

aum_df = handler.query(f"""
    SELECT cik, year, quarter, total AS aum
    FROM {TGT_CIK_AUM}
    WHERE total > 0
""")
aum_df["period"] = (aum_df["year"] * 4 + (aum_df["quarter"] - 1)).astype("int32")

# periods come from the (small) aum table — no full scan of normalized_holdings
periods = sorted(aum_df[["year", "quarter"]].drop_duplicates()
                       .itertuples(index=False, name=None))
print(f"aum_df: {len(aum_df):,} rows  |  {len(periods)} periods "
      f"({periods[0]} -> {periods[-1]})")


aum_df: 240,595 rows  |  49 periods ((2013, 2) -> (2025, 2))


### Streaming helpers

One quarter in memory at a time. Each helper takes a small DataFrame and returns a small reduction.

In [8]:
def load_quarter(y: int, q: int) -> pd.DataFrame:
    df = handler.query(f"""
        SELECT cik, cusip, weight
        FROM {TGT_NORMALIZED_HOLDINGS}
        WHERE year = {y} AND quarter = {q} AND weight IS NOT NULL AND weight > 0
    """)
    df["weight"] = df["weight"].astype("float32")
    return df

def quarter_concentration(cur: pd.DataFrame, period: int) -> pd.DataFrame:
    g = cur.groupby("cik", sort=False)["weight"]
    out = pd.DataFrame({
        "n_holdings": g.size().astype("int32"),
        "hhi": g.apply(lambda w: float((w.values ** 2).sum())),
        "top5_weight": g.apply(lambda w: float(np.sort(w.values)[-5:].sum())),
    }).reset_index()
    out["period"] = np.int32(period)
    return out

def pair_turnover(cur: pd.DataFrame, prev: pd.DataFrame, period: int) -> pd.DataFrame:
    m = cur[["cik", "cusip", "weight"]].merge(
        prev.rename(columns={"weight": "wp"}),
        on=["cik", "cusip"], how="outer",
    )
    m[["weight", "wp"]] = m[["weight", "wp"]].fillna(0.0)
    now, prior = m["weight"] > 0, m["wp"] > 0
    agg = (m.assign(ad=(m["weight"] - m["wp"]).abs(),
                    op=(now & ~prior).astype("int32"),
                    cl=(~now & prior).astype("int32"),
                    nn=now.astype("int32"), pp=prior.astype("int32"))
             .groupby("cik", as_index=False, sort=False)
             .agg(turnover=("ad", lambda x: 0.5 * x.sum()),
                  n_opened=("op", "sum"), n_closed=("cl", "sum"),
                  n_now=("nn", "sum"), n_prev=("pp", "sum")))
    agg = agg[agg["n_prev"] > 0].copy()
    agg["open_rate"] = agg["n_opened"] / agg["n_now"].clip(lower=1)
    agg["close_rate"] = agg["n_closed"] / agg["n_prev"].clip(lower=1)
    agg["period"] = np.int32(period)
    return agg[["cik", "period", "turnover", "open_rate", "close_rate"]]

def update_runs(runs: pd.DataFrame, cur: pd.DataFrame, period: int):
    """Extend/open (cik,cusip) runs; flush stale ones. Returns (new_runs, completed_df_or_None)."""
    completed = None
    if len(runs):
        stale = runs["last_seen"].values < (period - 1)
        if stale.any():
            s = runs.iloc[stale]
            completed = pd.DataFrame({
                "cik": s.index.get_level_values("cik"),
                "run_len": (s["last_seen"].values - s["run_start"].values + 1).astype("int32"),
            })
            runs = runs.iloc[~stale]
    cur_idx = pd.MultiIndex.from_arrays([cur["cik"].values, cur["cusip"].values],
                                        names=["cik", "cusip"])
    if len(runs):
        existing = cur_idx.intersection(runs.index)
        if len(existing):
            runs.loc[existing, "last_seen"] = np.int32(period)
        new_idx = cur_idx.difference(runs.index)
    else:
        new_idx = cur_idx
    if len(new_idx):
        new_rows = pd.DataFrame({"run_start": np.int32(period), "last_seen": np.int32(period)},
                                index=new_idx)
        runs = pd.concat([runs, new_rows]) if len(runs) else new_rows
    return runs, completed


### Stream quarters

Loop through each quarter, accumulate small per-quarter reductions, keep only the previous quarter + active runs in memory.

In [9]:
qstats_chunks, churn_chunks, run_chunks = [], [], []
runs = pd.DataFrame({"run_start": pd.Series(dtype="int32"),
                     "last_seen": pd.Series(dtype="int32")})
runs.index = pd.MultiIndex.from_arrays([[], []], names=["cik", "cusip"])
prev, prev_p, t0 = None, None, time.time()

for y, q in periods:
    p = y * 4 + (q - 1)
    cur = load_quarter(y, q)
    if cur.empty:
        prev = None
        continue
    qstats_chunks.append(quarter_concentration(cur, p))
    if prev is not None and prev_p == p - 1:
        churn_chunks.append(pair_turnover(cur, prev, p))
    runs, done = update_runs(runs, cur, p)
    if done is not None:
        run_chunks.append(done)
    prev, prev_p = cur[["cik", "cusip", "weight"]], p
    print(f"  p={p} ({y}Q{q}) rows={len(cur):,}  active_runs={len(runs):,}  t={time.time()-t0:.0f}s")

# flush remaining runs
if len(runs):
    run_chunks.append(pd.DataFrame({
        "cik": runs.index.get_level_values("cik"),
        "run_len": (runs["last_seen"].values - runs["run_start"].values + 1).astype("int32"),
    }))

quarter_stats = (pd.concat(qstats_chunks, ignore_index=True)
                   .merge(aum_df[["cik", "period", "aum"]], on=["cik", "period"], how="left"))
churn = pd.concat(churn_chunks, ignore_index=True)
duration = (pd.concat(run_chunks, ignore_index=True)
              .groupby("cik")["run_len"].mean().rename("avg_holding_duration"))

print(f"\nquarter_stats: {len(quarter_stats):,}  churn: {len(churn):,}  "
      f"CIKs: {quarter_stats['cik'].nunique():,}")


  p=8053 (2013Q2) rows=18,341  active_runs=18,341  t=0s
  p=8054 (2013Q3) rows=31,919  active_runs=26,743  t=0s
  p=8055 (2013Q4) rows=938,257  active_runs=583,340  t=3s
  p=8056 (2014Q1) rows=940,034  active_runs=662,942  t=8s
  p=8057 (2014Q2) rows=978,338  active_runs=679,137  t=12s
  p=8058 (2014Q3) rows=971,622  active_runs=685,973  t=18s
  p=8059 (2014Q4) rows=1,008,690  active_runs=725,899  t=24s
  p=8060 (2015Q1) rows=1,041,392  active_runs=737,005  t=29s
  p=8061 (2015Q2) rows=1,018,551  active_runs=738,809  t=35s
  p=8062 (2015Q3) rows=1,012,083  active_runs=742,995  t=41s
  p=8063 (2015Q4) rows=1,048,810  active_runs=758,138  t=46s
  p=8064 (2016Q1) rows=1,072,474  active_runs=765,806  t=51s
  p=8065 (2016Q2) rows=1,073,059  active_runs=771,014  t=57s
  p=8066 (2016Q3) rows=1,068,770  active_runs=771,457  t=62s
  p=8067 (2016Q4) rows=1,114,548  active_runs=809,906  t=67s
  p=8068 (2017Q1) rows=1,139,931  active_runs=820,980  t=73s
  p=8069 (2017Q2) rows=1,121,723  active_run

## Per-quarter per-CIK metrics

First pass: for each (cik, quarter) compute portfolio shape and turnover vs. the previous quarter.

In [ ]:
# `quarter_stats` (with aum merged) was produced in the streaming loader above.
# Preview:
print(f"quarter_stats: {len(quarter_stats):,} rows, {quarter_stats['cik'].nunique():,} CIKs")
quarter_stats.head()


## Turnover and churn between consecutive quarters

- `turnover = 0.5 * Σ|Δw|` over the union of positions in t−1 and t (missing side = weight 0).
- `open_rate` = fraction of current positions not present last quarter.
- `close_rate` = fraction of previous positions not present now (relative to previous n_holdings).

In [ ]:
# `churn` was produced in the streaming loader (per adjacent-quarter pair).
print(f"churn: {len(churn):,} rows")
churn.head()


## Average holding duration per CIK

For each (cik, cusip) we count the number of runs of consecutive quarters and their total length. The weighted-by-positions mean run length is the average time a position is held.

In [ ]:
# `duration` was computed incrementally via run-tracking in the streaming loader.
duration.describe()


## Aggregate everything into a per-CIK profile

In [ ]:
def build_cik_profile(qstats: pd.DataFrame, churn: pd.DataFrame, duration: pd.Series) -> pd.DataFrame:
    base = qstats.groupby("cik").agg(
        n_quarters=("period", "nunique"),
        first_period=("period", "min"),
        last_period=("period", "max"),
        aum_mean=("aum", "mean"),
        aum_median=("aum", "median"),
        n_holdings_mean=("n_holdings", "mean"),
        n_holdings_std=("n_holdings", "std"),
        hhi_mean=("hhi", "mean"),
        top5_weight_mean=("top5_weight", "mean"),
    )

    # AUM dynamics
    aum_sorted = qstats.dropna(subset=["aum"]).sort_values(["cik", "period"])
    log_aum = np.log(aum_sorted["aum"])
    aum_sorted = aum_sorted.assign(log_aum=log_aum)
    aum_log_std = aum_sorted.groupby("cik")["log_aum"].std().rename("aum_log_std")

    first_last = aum_sorted.groupby("cik").agg(
        aum_first=("aum", "first"),
        aum_last=("aum", "last"),
        p_first=("period", "first"),
        p_last=("period", "last"),
    )
    dt_years = ((first_last["p_last"] - first_last["p_first"]) / 4).clip(lower=0.25)
    first_last["aum_cagr"] = (first_last["aum_last"] / first_last["aum_first"]) ** (1 / dt_years) - 1

    churn_agg = churn.groupby("cik").agg(
        turnover_mean=("turnover", "mean"),
        turnover_std=("turnover", "std"),
        open_rate_mean=("open_rate", "mean"),
        close_rate_mean=("close_rate", "mean"),
    )

    out = base.join(aum_log_std).join(first_last[["aum_cagr"]]).join(churn_agg).join(duration)
    return out.reset_index()

profile = build_cik_profile(quarter_stats, churn, duration)
print(f"profile rows: {len(profile):,}")
profile.head()

## Distributions

Eyeball each metric to pick sensible threshold ranges.

In [ ]:
plot_cols = [
    "n_quarters", "aum_mean", "n_holdings_mean", "hhi_mean",
    "top5_weight_mean", "turnover_mean", "open_rate_mean",
    "close_rate_mean", "avg_holding_duration", "aum_log_std", "aum_cagr",
]
log_cols = {"aum_mean", "n_holdings_mean"}

fig, axes = plt.subplots(4, 3, figsize=(15, 14))
for ax, col in zip(axes.flat, plot_cols):
    data = profile[col].dropna()
    if col in log_cols:
        data = data[data > 0]
        ax.hist(np.log10(data), bins=60, color="steelblue", edgecolor="black")
        ax.set_xlabel(f"log10({col})")
    else:
        ax.hist(data, bins=60, color="steelblue", edgecolor="black")
        ax.set_xlabel(col)
    ax.set_title(col)
for ax in axes.flat[len(plot_cols):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(profile[plot_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T)

## Archetype heuristics

Derived flags to label CIKs by behavior. Cutoffs use distribution percentiles so they adapt to the data.

In [ ]:
def tag_archetypes(p: pd.DataFrame) -> pd.DataFrame:
    q = p.copy()
    turnover_low = q["turnover_mean"].quantile(0.25)
    turnover_high = q["turnover_mean"].quantile(0.75)
    duration_long = q["avg_holding_duration"].quantile(0.75)
    churn_high = q["open_rate_mean"].quantile(0.75)
    conc_high = q["hhi_mean"].quantile(0.75)
    conc_low = q["hhi_mean"].quantile(0.25)

    q["is_buy_and_hold"] = (q["turnover_mean"] <= turnover_low) & (q["avg_holding_duration"] >= duration_long)
    q["is_high_churn"] = (q["turnover_mean"] >= turnover_high) | (q["open_rate_mean"] >= churn_high)
    q["is_concentrated"] = q["hhi_mean"] >= conc_high
    q["is_diversified"] = q["hhi_mean"] <= conc_low
    return q

profile = tag_archetypes(profile)
profile[["is_buy_and_hold", "is_high_churn", "is_concentrated", "is_diversified"]].sum()

## ⚙️  Filter — tune thresholds here

Set any criterion to `None` to skip it. The `filter_ciks` function returns a DataFrame of matching CIKs, ranked by AUM.

In [ ]:
def filter_ciks(
    p: pd.DataFrame,
    min_quarters: int | None = 8,
    min_aum: float | None = None,
    max_aum: float | None = None,
    min_n_holdings: int | None = None,
    max_n_holdings: int | None = None,
    min_hhi: float | None = None,
    max_hhi: float | None = None,
    min_turnover: float | None = None,
    max_turnover: float | None = None,
    min_avg_duration: float | None = None,
    max_avg_duration: float | None = None,
    max_open_rate: float | None = None,
    archetype: str | None = None,  # one of: 'buy_and_hold', 'high_churn', 'concentrated', 'diversified'
) -> pd.DataFrame:
    """Return CIK profiles matching the given thresholds, sorted by AUM desc."""
    q = p.copy()
    mask = pd.Series(True, index=q.index)
    if min_quarters is not None: mask &= q["n_quarters"] >= min_quarters
    if min_aum is not None: mask &= q["aum_mean"] >= min_aum
    if max_aum is not None: mask &= q["aum_mean"] <= max_aum
    if min_n_holdings is not None: mask &= q["n_holdings_mean"] >= min_n_holdings
    if max_n_holdings is not None: mask &= q["n_holdings_mean"] <= max_n_holdings
    if min_hhi is not None: mask &= q["hhi_mean"] >= min_hhi
    if max_hhi is not None: mask &= q["hhi_mean"] <= max_hhi
    if min_turnover is not None: mask &= q["turnover_mean"] >= min_turnover
    if max_turnover is not None: mask &= q["turnover_mean"] <= max_turnover
    if min_avg_duration is not None: mask &= q["avg_holding_duration"] >= min_avg_duration
    if max_avg_duration is not None: mask &= q["avg_holding_duration"] <= max_avg_duration
    if max_open_rate is not None: mask &= q["open_rate_mean"] <= max_open_rate
    if archetype is not None:
        col = f"is_{archetype}"
        if col not in q.columns:
            raise ValueError(f"unknown archetype '{archetype}'")
        mask &= q[col]
    return q.loc[mask].sort_values("aum_mean", ascending=False).reset_index(drop=True)

# ---- Example: long-lived, medium-sized, low-turnover buy-and-hold investors ----
selected = filter_ciks(
    profile,
    min_quarters=12,
    min_aum=1e8,
    max_turnover=0.15,
    min_avg_duration=4.0,
)
print(f"matched CIKs: {len(selected):,}")
selected.head(20)

In [ ]:
# Export the matching CIK list
cik_list = selected["cik"].tolist()
print(f"{len(cik_list)} CIKs")
print(cik_list[:20])

## Quick archetype counts

In [ ]:
for arch in ["buy_and_hold", "high_churn", "concentrated", "diversified"]:
    sub = filter_ciks(profile, min_quarters=8, archetype=arch)
    print(f"{arch:16s}  n={len(sub):5d}  median AUM={sub['aum_mean'].median():.2e}  median turnover={sub['turnover_mean'].median():.3f}")